# Fine-tune the car classifier on a GPU (road-view upgrade)

Fine-tunes the same CLIP ViT-B/16 image model the web app uses, on the 21k crawl crops plus the
hand-labelled dashcam crops, with augmentations that imitate dashcam footage (blur, low resolution,
partial occlusion). Scores the held-out KL traffic-jam cars before and after training.

**Steps**
1. *Runtime → Change runtime type → T4 GPU* (free tier is fine).
2. *Runtime → Run all*. When asked, allow access to Google Drive.
3. About 30–40 minutes later, `Car Brand Recognition/finetuned/` in your Drive has
   `clip_ft_visual_fp16.pt`, `head_ft.json` and `results_ft.json`.

Expects `car_gpu_data.zip` in the `Car Brand Recognition` folder of your Drive.

In [ ]:
EPOCHS      = 6
BATCH       = 64
UNFREEZE    = 6        # last N transformer blocks that are trained (of 12)
LR_BACKBONE = 1e-5
LR_HEAD     = 2e-4
ROAD_SHARE  = 0.20     # share of each batch drawn from the hand-labelled dashcam crops
DRIVE_DIR   = "/content/drive/MyDrive/Car Brand Recognition"
SEED        = 0

In [ ]:
!pip -q install open_clip_torch==2.24.0
import os, zipfile, json, time, random
from google.colab import drive
drive.mount("/content/drive")
zp = f"{DRIVE_DIR}/car_gpu_data.zip"
assert os.path.exists(zp), f"car_gpu_data.zip not found in {DRIVE_DIR}"
if not os.path.exists("/content/data/head_init.json"):
    t = time.time(); zipfile.ZipFile(zp).extractall("/content/data"); print(f"unzipped in {time.time()-t:.0f}s")
os.makedirs(f"{DRIVE_DIR}/finetuned", exist_ok=True)

In [ ]:
import glob, numpy as np, torch, torch.nn as nn, torch.nn.functional as F, open_clip
from PIL import Image
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as T
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
dev = "cuda"; assert torch.cuda.is_available(), "Turn on the GPU: Runtime -> Change runtime type -> T4 GPU"
D = "/content/data"
head = json.load(open(f"{D}/head_init.json")); classes = head["classes"]; ci = {c: i for i, c in enumerate(classes)}
crawl = [(p, ci[p.split('/')[-2]]) for p in glob.glob(f"{D}/train/*/*.jpg")]
road  = [(p, ci[p.split('/')[-2]]) for p in glob.glob(f"{D}/road/*/*.jpg")]
print(len(classes), "classes,", len(crawl), "crawl crops,", len(road), "road crops")
MEAN, STD = (0.48145466, 0.4578275, 0.40821073), (0.26862954, 0.26130258, 0.27577711)
class LowRes:
    def __call__(self, im):
        if random.random() < 0.35:
            s = random.randint(40, 110); w, h = im.size; r = s / max(w, h)
            im = im.resize((max(8, int(w*r)), max(8, int(h*r))), Image.BILINEAR).resize((w, h), Image.BILINEAR)
        return im
train_tf = T.Compose([LowRes(), T.RandomResizedCrop(224, scale=(0.5, 1.0), ratio=(0.6, 1.8), interpolation=T.InterpolationMode.BICUBIC),
    T.RandomHorizontalFlip(), T.ColorJitter(0.35, 0.35, 0.25, 0.03),
    T.RandomApply([T.GaussianBlur(5, (0.1, 2.0))], p=0.3), T.ToTensor(), T.Normalize(MEAN, STD), T.RandomErasing(p=0.25, scale=(0.02, 0.2))])
eval_tf = T.Compose([T.Resize(224, interpolation=T.InterpolationMode.BICUBIC), T.CenterCrop(224), T.ToTensor(), T.Normalize(MEAN, STD)])
class DS(Dataset):
    def __init__(s, items, tf): s.items, s.tf = items, tf
    def __len__(s): return len(s.items)
    def __getitem__(s, i):
        p, y = s.items[i]; return s.tf(Image.open(p).convert("RGB")), y
items = crawl + road
w = np.concatenate([np.full(len(crawl), (1-ROAD_SHARE)/len(crawl)), np.full(len(road), ROAD_SHARE/len(road))])
sampler = WeightedRandomSampler(torch.tensor(w, dtype=torch.double), num_samples=len(crawl), replacement=True)
train_dl = DataLoader(DS(items, train_tf), batch_size=BATCH, sampler=sampler, num_workers=2, pin_memory=True, drop_last=True)

In [ ]:
model, _, _ = open_clip.create_model_and_transforms("ViT-B-16", pretrained="laion2b_s34b_b88k")
visual = model.visual.to(dev); del model
class Net(nn.Module):
    def __init__(s):
        super().__init__(); s.v = visual; s.fc = nn.Linear(512, len(classes))
        s.fc.weight.data = torch.tensor(head["coef"], dtype=torch.float32); s.fc.bias.data = torch.tensor(head["intercept"], dtype=torch.float32)
    def forward(s, x):
        e = s.v(x); e = e / e.norm(dim=-1, keepdim=True); return s.fc(e)
net = Net().to(dev)
for p in net.v.parameters(): p.requires_grad = False
blocks = net.v.transformer.resblocks
train_mods = list(blocks[-UNFREEZE:]) + [net.v.ln_post]
for m in train_mods:
    for p in m.parameters(): p.requires_grad = True
net.v.proj.requires_grad = True
bb = [p for p in net.v.parameters() if p.requires_grad]
opt = torch.optim.AdamW([{"params": bb, "lr": LR_BACKBONE}, {"params": net.fc.parameters(), "lr": LR_HEAD}], weight_decay=0.05)
steps = EPOCHS * len(train_dl)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[LR_BACKBONE, LR_HEAD], total_steps=steps, pct_start=0.1)
scaler = torch.cuda.amp.GradScaler()
print(f"training {sum(p.numel() for p in bb)/1e6:.1f}M backbone params, {steps} steps")

In [ ]:
@torch.no_grad()
def probs(paths):
    net.eval(); out = []
    for k in range(0, len(paths), 128):
        x = torch.stack([eval_tf(Image.open(p).convert("RGB")) for p in paths[k:k+128]]).to(dev)
        with torch.autocast("cuda"): out.append(F.softmax(net(x).float(), 1).cpu())
    return torch.cat(out)
def score_photos(root):
    fs = sorted(glob.glob(f"{D}/{root}/*/*.jpg")); y = [f.split('/')[-2] for f in fs]
    P = probs(fs); pred = [classes[i] for i in P.argmax(1)]
    return float(np.mean([a == b for a, b in zip(pred, y)]))
man = json.load(open(f"{D}/test_road.json")); tracks = {}
for m in man: tracks.setdefault(m["track"], (m["label"], []))[1].append(f'{D}/{m["file"]}')
def score_road():
    allf = [f for _, fs in tracks.values() for f in fs]; P = probs(allf); i = 0; right = cr = cw = 0
    for lab, fs in tracks.values():
        m = P[i:i+len(fs)].mean(0); i += len(fs); j = int(m.argmax()); ok = classes[j] == lab; pj = float(m[j])
        right += int(ok); cr += int(ok and pj >= 0.30); cw += int((not ok) and pj >= 0.30)
    return {"cars": len(tracks), "top1": right, "shown_right": cr, "shown_wrong": cw}
def evaluate(tag):
    r = {"road": score_road(), "commons": score_photos("test_commons_v2"), "openverse": score_photos("test_openverse")}
    print(f"[{tag}] road {r['road']}  commons {r['commons']:.1%}  openverse {r['openverse']:.1%}"); return r
history = {"before": evaluate("before training")}

In [ ]:
def save(tag, r):
    sd = {k: v.half().cpu() for k, v in net.v.state_dict().items()}
    torch.save(sd, f"{DRIVE_DIR}/finetuned/clip_ft_visual_fp16.pt")
    json.dump({"classes": classes, "labels": classes, "coef": net.fc.weight.detach().cpu().numpy().round(5).tolist(),
               "intercept": net.fc.bias.detach().cpu().numpy().round(5).tolist(), "arch": "ViT-B-16 fine-tuned",
               "weights": "laion2b_s34b_b88k"}, open(f"{DRIVE_DIR}/finetuned/head_ft.json", "w"))
    print("   saved", tag)
key = lambda r: r["road"]["top1"] / r["road"]["cars"] + r["road"]["shown_right"] / r["road"]["cars"] - r["road"]["shown_wrong"] / r["road"]["cars"] + r["commons"]
best = key(history["before"]); crit = nn.CrossEntropyLoss(label_smoothing=0.1)
for ep in range(EPOCHS):
    net.train(); t = time.time(); tot = 0
    for b, (x, y) in enumerate(train_dl):
        x, y = x.to(dev, non_blocking=True), y.to(dev, non_blocking=True)
        with torch.autocast("cuda"): loss = crit(net(x), y)
        opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        tot += loss.item()
        if b % 100 == 0: print(f"  epoch {ep+1} step {b}/{len(train_dl)} loss {tot/(b+1):.3f}")
    r = evaluate(f"epoch {ep+1}, {time.time()-t:.0f}s"); history[f"epoch{ep+1}"] = r
    if key(r) > best: best = key(r); save(f"epoch {ep+1}", r); history["best"] = f"epoch{ep+1}"
json.dump(history, open(f"{DRIVE_DIR}/finetuned/results_ft.json", "w"), indent=1)
print("best:", history.get("best", "none - training did not beat the starting model"))